# 04 — Experiments: Full Paper Results

Runs all 7 experiments to produce publication figures and tables.

1. Model Comparison Table (5-fold CV)
2. Feature Ablation
3. Per-Type Detection Rates
4. Hyperparameter Sensitivity
5. Detection Examples
6. Feature Importance
7. Statistical Significance

**Input:** `data/labeled/` from notebook 01
**Output:** `results/figures/*.pdf`, `results/tables/*.tex`, `results/raw/*.json`

In [ ]:
!pip install -q lightkurve astroquery PyWavelets matplotlib

import sys, os, glob

# Auto-detect codebase path
CODEBASE_PATH = None
for candidate in [
    '/kaggle/input/exopattern-codebase',
    *glob.glob('/kaggle/input/exopattern-codebase/*/'),
    *glob.glob('/kaggle/input/exopattern*/'),
    *glob.glob('/kaggle/input/exopattern*/*/'),
]:
    if os.path.isdir(os.path.join(candidate, 'backend')):
        CODEBASE_PATH = candidate.rstrip('/')
        break

if CODEBASE_PATH is None:
    raise FileNotFoundError(
        "Could not find 'backend/' directory in /kaggle/input/. "
        "Upload the project root (containing backend/ and experiments/) as a Kaggle dataset."
    )

sys.path.insert(0, CODEBASE_PATH)
print(f"Codebase found at: {CODEBASE_PATH}")

import numpy as np
import json
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# Auto-detect data directory
DATA_DIR = None
for candidate in [
    '/kaggle/input/exopattern-labeled-data/data/labeled',
    *glob.glob('/kaggle/input/exopattern-labeled*/data/labeled'),
    *glob.glob('/kaggle/input/exopattern-labeled*/*/data/labeled'),
    '/kaggle/working/data/labeled',
    'data/labeled',
]:
    if os.path.exists(os.path.join(candidate, 'metadata.csv')):
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError("Could not find labeled dataset (metadata.csv) in /kaggle/input/")

print(f"Using data from: {DATA_DIR}")

In [ ]:
# Override output paths for Kaggle
import experiments.config as config
config.RESULTS_DIR = '/kaggle/working/results'
config.FIGURES_DIR = '/kaggle/working/results/figures'
config.TABLES_DIR = '/kaggle/working/results/tables'
config.RAW_DIR = '/kaggle/working/results/raw'

from experiments.run_experiments import ExperimentRunner

runner = ExperimentRunner(
    data_dir=DATA_DIR,
    results_dir='/kaggle/working/results'
)

## Experiment 1: Model Comparison Table

All classical models × 5-fold stratified CV → precision, recall, F1, ROC-AUC, PR-AUC with std.

In [ ]:
results_1 = runner.experiment_1_model_comparison()

print("\n=== Model Comparison Results ===")
for model_name, metrics in results_1.items():
    print(f"{model_name:25s}: F1={metrics['f1_mean']:.3f}±{metrics['f1_std']:.3f}  "
          f"P={metrics['precision_mean']:.3f}  R={metrics['recall_mean']:.3f}  "
          f"ROC-AUC={metrics['roc_auc_mean']:.3f}")

## Experiment 2: Feature Ablation

Ensemble model with cumulative feature groups: statistical → +frequency → +wavelet → +autocorrelation → +shape.

In [ ]:
results_2 = runner.experiment_2_feature_ablation()

print("\n=== Feature Ablation Results ===")
for label, f1 in results_2.items():
    print(f"{label:25s}: F1={f1:.3f}")

## Experiment 3: Per-Type Detection Rates

In [ ]:
results_3 = runner.experiment_3_per_type_detection()

print("\n=== Per-Type Detection ===")
for atype, metrics in results_3.items():
    print(f"{atype}: {metrics}")

## Experiment 4: Hyperparameter Sensitivity

In [ ]:
results_4 = runner.experiment_4_hyperparameter_sensitivity()

print("\n=== Contamination Sweep ===")
for c, f1 in results_4['contamination'].items():
    print(f"  contamination={c}: F1={f1:.3f}")

print("\n=== Window Size Sweep ===")
for ws, f1 in results_4['window_size'].items():
    print(f"  window_size={ws}: F1={f1:.3f}")

## Experiment 5: Detection Examples

In [ ]:
results_5 = runner.experiment_5_detection_examples()
print(f"Generated {len(results_5)} detection example plots")

## Experiment 6: Feature Importance

In [ ]:
results_6 = runner.experiment_6_feature_importance()

# Print top 10
sorted_features = sorted(results_6.items(), key=lambda x: x[1], reverse=True)
print("\n=== Top 10 Most Important Features ===")
for name, imp in sorted_features[:10]:
    print(f"  {name:30s}: {imp:.4f}")

## Experiment 7: Statistical Significance

In [ ]:
results_7 = runner.experiment_7_statistical_significance()

print("\n=== Pairwise Statistical Tests ===")
for pair, test in results_7.items():
    sig = '***' if test['ttest_pvalue'] < 0.001 else '**' if test['ttest_pvalue'] < 0.01 else '*' if test['ttest_pvalue'] < 0.05 else 'ns'
    print(f"  {pair:50s}: diff={test['mean_diff']:+.4f}  p={test['ttest_pvalue']:.4f} {sig}")

## Output Summary

In [ ]:
from pathlib import Path

print("=== Generated Files ===")
for subdir in ['figures', 'tables', 'raw']:
    d = Path(f'/kaggle/working/results/{subdir}')
    if d.exists():
        print(f"\n{subdir}/")
        for f in sorted(d.glob('*')):
            size_kb = f.stat().st_size / 1024
            print(f"  {f.name}: {size_kb:.1f} KB")

## Done!

Download `results/` from this notebook's output:
- `results/figures/*.pdf` — publication figures
- `results/tables/*.tex` — LaTeX tables
- `results/raw/*.json` — raw metrics for further analysis

Place in `artifacts/results/` locally.

## Package Results as ZIP

In [ ]:
import shutil

# Zip all experiment results (figures, tables, raw JSON)
shutil.make_archive('/kaggle/working/exopattern-results', 'zip', '/kaggle/working/results')
zip_size = os.path.getsize('/kaggle/working/exopattern-results.zip') / (1024 * 1024)
print(f"Created exopattern-results.zip ({zip_size:.1f} MB)")
print("Download and place contents in artifacts/results/ locally.")
print("Contains: figures/*.pdf, tables/*.tex, raw/*.json")